# 02. Semantic and String Distances

**Goal:** Compute advanced distance metrics: TF-IDF Cosine Similarity and Character-level String Match Ratio (similar to Levenshtein distance).

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import paired_cosine_distances
import difflib

## 1. Load Data

In [2]:
df = pd.read_csv('../data/processed/features_part1.csv')
df['q1_clean'] = df['q1_clean'].fillna('')
df['q2_clean'] = df['q2_clean'].fillna('')

> **📌 Decision Note — Why Semantic Similarity Metric?**
>
> **Chosen approach:** TF-IDF Cosine Similarity
>
> **Why this works:** Cosine similarity measures the angle between two vectors. It perfectly captures how semantically similar two sentences are, completely ignoring document length differences.
>
> **Alternatives we could have used:**
> | Option | Pros | Cons |
> |--------|------|------|
> | Euclidean Distance | Simple spatial distance | Heavily penalized by length. A short sentence and a long sentence with similar meaning will have a huge Euclidean distance. |
> | Manhattan Distance | Grid-based distance | Same issue as Euclidean, highly sensitive to magnitude/length of the document. |
>
> **Why we chose this over alternatives:** Cosine similarity normalizes for magnitude, making it the mathematical gold-standard for vector similarity.

## 2. Compute TF-IDF Cosine Similarity

In [3]:
# Fit a single TF-IDF on all questions to build a shared vocabulary
all_questions = pd.concat([df['q1_clean'], df['q2_clean']])
tfidf = TfidfVectorizer(max_features=10000)
tfidf.fit(all_questions)

# Transform Q1 and Q2 separately
tfidf_q1 = tfidf.transform(df['q1_clean'])
tfidf_q2 = tfidf.transform(df['q2_clean'])

# Compute pairwise cosine *distance* (1 - similarity)
# Similarity = 1 - Distance
cos_distances = paired_cosine_distances(tfidf_q1, tfidf_q2)
df['tfidf_cosine_sim'] = 1.0 - cos_distances

## 3. Compute Character-Level String Ratio

In [4]:
def string_ratio(row):
    s1 = str(row['q1_clean'])
    s2 = str(row['q2_clean'])
    return difflib.SequenceMatcher(None, s1, s2).ratio()

df['string_ratio'] = df.apply(string_ratio, axis=1)

df[['jaccard_sim', 'tfidf_cosine_sim', 'string_ratio', 'is_duplicate']].head()

,jaccard_sim,tfidf_cosine_sim,string_ratio,is_duplicate
0,0.105263,0.034266,0.388889,0
1,0.882353,0.819097,0.899329,0
2,0.090909,0.506039,0.566038,0
3,0.571429,0.848463,0.685185,1
4,0.200000,0.484310,0.462222,0


In [5]:
df.to_csv('../data/processed/final_features.csv', index=False)
print('Saved final features.')

Saved final features.
